# Fair Ablation: Laplacian vs Edge/Freq (Both 30 Epochs)

## Why "Fair"?

The prior ablation compared Laplacian at 30 epochs vs Edge/Freq at 60 epochs. That comparison is confounded: any difference in rollout behaviour could come from extra training time rather than the loss function itself.

Here everything is held constant except the loss function:

| Factor | Laplacian Model | Edge/Freq Model |
|--------|----------------|-----------------|
| Architecture | FluidWorldModelV2 (d_model=128) | FluidWorldModelV2 (d_model=128) |
| Training epochs | **30** | **30** |
| Training data | Moving MNIST | Moving MNIST |
| Base loss | BCE | BCE |
| Laplacian regularization | Yes | Yes |
| Edge loss (`edge_weight`) | No | **0.01** |
| Frequency loss (`freq_weight`) | No | **0.005** |

The only difference is the addition of edge and frequency loss terms. If the Edge/Freq model still shows different autopoietic recovery behaviour under these conditions, the effect is attributable to the loss terms themselves.

## Structure

1. Imports and setup
2. Model loading (30-epoch Edge/Freq checkpoint)
3. Autoregressive rollout: 500 paired rollouts, SSIM and MSE
4. Laplacian baseline loading (pre-computed, same 500 sequences)
5. Statistical comparison: paired t-test, Wilcoxon, Cohen's d
6. Publication figures: SSIM curves with CIs, recovery distributions
7. Sample rollout visualisation: side-by-side frame comparisons
8. Save results

## 1. Imports and Environment Setup

PyTorch, NumPy, SciPy (paired t-test and Wilcoxon), Matplotlib, and `FluidWorldModelV2`. Auto-detects GPU.

GPU rollouts take ~2-3 minutes. CPU: 10-30 minutes. If you have a GPU but see `cpu`, check your PyTorch CUDA install.

In [ ]:
import sys, time
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats

PROJECT = Path(r"C:\DEV\Workspace\active\coding\_AI RESEARCH\FluidWorld")
sys.path.insert(0, str(PROJECT))

from fluidworld.core.world_model_v2 import FluidWorldModelV2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 2. Load Model Checkpoint and Evaluation Data

Two steps:

1. **Build and load the Edge/Freq model.** `FluidWorldModelV2` with training-time hyperparameters (d_model=128, 3 encoder layers, 16x16 belief resolution, all auxiliary modules on). 30-epoch checkpoint loaded, model set to `eval()`.

2. **Load and subsample evaluation data.** Full Moving MNIST test set (10,000 sequences, 20 frames, 64x64 grayscale). Seed `RandomState(42)` selects 500 sequences, same ones used for the Laplacian evaluation. Required for valid paired tests.

If `load_state_dict` throws a `RuntimeError` about missing keys, the checkpoint was saved with a different architecture config.

In [ ]:
# === CONFIGURATION ===
DATA_PATH = str(PROJECT / "data" / "mnist_test_seq.npy")
CKPT_EDGEFREQ_FAIR = str(PROJECT / "checkpoints" / "moving_mnist_edgefreq_fair" / "model_epoch_30.pt")

N_ROLLOUTS = 500
ROLLOUT_STEPS = 19
BATCH_SIZE = 50

# === BUILD MODEL ===
model = FluidWorldModelV2(
    in_channels=1, d_model=128, stimulus_dim=1,
    n_encoder_layers=3, max_steps_encoder=6,
    belief_spatial_hw=16, n_belief_evolve=3,
    loss_type="bce", use_fatigue=False, use_inhibition=True,
    use_memory_pump=True, use_hebbian=True, use_deltanet=True, use_titans=True,
).to(device)

ckpt = torch.load(CKPT_EDGEFREQ_FAIR, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f"Loaded Edge/Freq FAIR checkpoint (30 epochs)")
print(f"  Checkpoint keys: {list(ckpt.keys())}")

# === LOAD DATA ===
print("\nLoading Moving MNIST...")
data = np.load(DATA_PATH)
if data.shape[0] == 20 and data.shape[1] == 10000:
    data = data.transpose(1, 0, 2, 3)
data = data.astype(np.float32)
if data.max() > 1.0:
    data = data / 255.0

# Same sequences as the Laplacian evaluation
rng = np.random.RandomState(42)
indices = rng.choice(data.shape[0], size=N_ROLLOUTS, replace=False)
eval_data = data[indices]
print(f"Selected {N_ROLLOUTS} sequences, shape: {eval_data.shape}")
print(f"Value range: [{eval_data.min():.3f}, {eval_data.max():.3f}]")

## 3. Run 500 Autoregressive Rollouts

The model gets ground-truth frame 0, then predicts frames 1-19 autoregressively (each prediction feeds the next step, no ground-truth correction). Errors compound, so the degradation rate reveals how well the model maintains internal coherence.

Two metrics per step:
- **SSIM** (Structural Similarity): perceptual quality, range [0,1]. Primary metric.
- **MSE**: included for completeness.

First 8 sequences' predictions cached in `sample_preds` for the visual comparison later. Runtime ~2-3 min on GPU (batch size 50, 10 batches).

Monotonically decreasing SSIM = steady degradation. Non-monotonic (dip then partial recovery) = autopoietic self-correction. That is what we are looking for.

In [ ]:
def compute_ssim_batch(pred, target):
    C1, C2 = 0.01**2, 0.03**2
    mu_p = F.avg_pool2d(pred, 3, 1, 1)
    mu_t = F.avg_pool2d(target, 3, 1, 1)
    sigma_p = F.avg_pool2d(pred**2, 3, 1, 1) - mu_p**2
    sigma_t = F.avg_pool2d(target**2, 3, 1, 1) - mu_t**2
    sigma_pt = F.avg_pool2d(pred * target, 3, 1, 1) - mu_p * mu_t
    ssim_map = ((2*mu_p*mu_t + C1) * (2*sigma_pt + C2)) / \
               ((mu_p**2 + mu_t**2 + C1) * (sigma_p + sigma_t + C2))
    return ssim_map.mean(dim=(1, 2, 3))


ssim_matrix = np.zeros((N_ROLLOUTS, ROLLOUT_STEPS))
mse_matrix = np.zeros((N_ROLLOUTS, ROLLOUT_STEPS))
n_batches = (N_ROLLOUTS + BATCH_SIZE - 1) // BATCH_SIZE
t_start = time.time()

# Store predictions for sample visualization later
sample_preds = None

with torch.no_grad():
    for batch_idx in range(n_batches):
        start = batch_idx * BATCH_SIZE
        end = min(start + BATCH_SIZE, N_ROLLOUTS)
        B = end - start

        batch = torch.from_numpy(eval_data[start:end]).unsqueeze(2).to(device)
        x_init = batch[:, 0]
        stim = torch.zeros(B, 1, device=device)

        rollout = model.rollout(x_init, stim, n_steps=ROLLOUT_STEPS)

        # Save first batch predictions for visualization
        if batch_idx == 0:
            sample_preds = rollout[:8].cpu().numpy()

        for t in range(ROLLOUT_STEPS):
            pred_t = rollout[:, t]
            gt_t = batch[:, t + 1]
            ssim_vals = compute_ssim_batch(pred_t, gt_t).cpu().numpy()
            mse_vals = ((pred_t - gt_t) ** 2).mean(dim=(1, 2, 3)).cpu().numpy()
            ssim_matrix[start:end, t] = ssim_vals
            mse_matrix[start:end, t] = mse_vals

        if (batch_idx + 1) % 2 == 0 or batch_idx == n_batches - 1:
            elapsed = time.time() - t_start
            print(f"  Batch {batch_idx+1}/{n_batches} | {elapsed:.1f}s")

elapsed = time.time() - t_start
print(f"\nDone in {elapsed:.1f}s")
print(f"SSIM matrix shape: {ssim_matrix.shape}")
print(f"MSE matrix shape:  {mse_matrix.shape}")
print(f"Mean SSIM step 1: {ssim_matrix[:, 0].mean():.4f}")
print(f"Mean SSIM step 19: {ssim_matrix[:, -1].mean():.4f}")

# Free GPU memory
del model
torch.cuda.empty_cache()

## 4. Load Pre-Computed Laplacian Baseline

SSIM and MSE matrices from the Laplacian-only model evaluation (`experiments/analysis/autopoietic_recovery_stats.npz`). Same 500 sequences (seed=42), same 19-step rollout, same metrics.

Why pre-computed instead of re-running:
1. **Reproducibility.** The saved `.npz` is the canonical Laplacian result. Re-running could introduce subtle GPU numerics differences.
2. **Valid pairing.** Row `i` in both matrices = same input sequence. Paired tests require this.
3. **Speed.** No need to reload a second model.

If the shape assertion fails, the Laplacian evaluation was run with different `N_ROLLOUTS` or `ROLLOUT_STEPS` and needs re-running.

In [ ]:
lap_path = PROJECT / "experiments" / "analysis" / "autopoietic_recovery_stats.npz"
lap_data = np.load(str(lap_path))

ssim_laplacian = lap_data['ssim_matrix']
mse_laplacian = lap_data['mse_matrix']

print(f"Laplacian results loaded from: {lap_path}")
print(f"  SSIM shape: {ssim_laplacian.shape}")
print(f"  MSE shape:  {mse_laplacian.shape}")
print(f"  Mean SSIM step 1:  {ssim_laplacian[:, 0].mean():.4f}")
print(f"  Mean SSIM step 19: {ssim_laplacian[:, -1].mean():.4f}")

assert ssim_laplacian.shape == ssim_matrix.shape, \
    f"Shape mismatch: Laplacian {ssim_laplacian.shape} vs EdgeFreq {ssim_matrix.shape}"
print("\nShape check passed: both models evaluated on same sequences.")

## 5. Statistical Comparison

### 5a. Per-Step SSIM
Mean SSIM at every rollout step for both models, plus the delta. Shows where in the rollout the models diverge.

### 5b. Recovery Analysis
Recovery magnitude per sequence = `max(SSIM after trough) - trough SSIM`. Recovery > 0.01 counts as a "recovery event." High rates point to self-correcting internal dynamics.

### 5c. Paired Tests on Final-Step SSIM

- **Paired t-test**: is the mean final-step SSIM difference significantly nonzero?
- **Wilcoxon signed-rank**: non-parametric alternative, robust to non-normal distributions. Use as primary test if the difference distribution looks skewed.
- **Cohen's d** (paired): effect size, independent of sample size. `mean(diffs) / std(diffs)`. |d| < 0.2 negligible, 0.2-0.5 small, 0.5-0.8 medium, >= 0.8 large.

With N=500, even negligible effects can reach statistical significance. Always check Cohen's d alongside p-values.

In [ ]:
steps = np.arange(1, ROLLOUT_STEPS + 1)

# --- Per-step statistics ---
lap_mean = ssim_laplacian.mean(axis=0)
lap_std = ssim_laplacian.std(axis=0)
lap_ci95 = 1.96 * lap_std / np.sqrt(N_ROLLOUTS)

ef_mean = ssim_matrix.mean(axis=0)
ef_std = ssim_matrix.std(axis=0)
ef_ci95 = 1.96 * ef_std / np.sqrt(N_ROLLOUTS)

# --- Recovery detection ---
def compute_recovery(mat):
    """For each sequence: recovery = max(SSIM after min) - min(SSIM)."""
    N, T = mat.shape
    min_step = np.argmin(mat, axis=1)
    min_ssim = mat[np.arange(N), min_step]
    max_after = np.zeros(N)
    for i in range(N):
        if min_step[i] < T - 1:
            max_after[i] = mat[i, min_step[i]+1:].max()
        else:
            max_after[i] = min_ssim[i]
    return max_after - min_ssim

recovery_lap = compute_recovery(ssim_laplacian)
recovery_ef = compute_recovery(ssim_matrix)

# --- Statistical tests on final-step SSIM ---
final_lap = ssim_laplacian[:, -1]
final_ef = ssim_matrix[:, -1]

t_stat, t_pval = stats.ttest_rel(final_lap, final_ef)
w_stat, w_pval = stats.wilcoxon(final_lap, final_ef)

# Cohen's d (paired)
diff = final_lap - final_ef
cohens_d = diff.mean() / diff.std()

# --- Print results ---
print("=" * 65)
print("  FAIR ABLATION: Laplacian (30ep) vs Edge/Freq (30ep)")
print("=" * 65)

print(f"\nStep-by-step SSIM comparison:")
print(f"{'Step':>5} {'Laplacian':>12} {'Edge/Freq':>12} {'Delta':>10}")
print("-" * 45)
for t in range(ROLLOUT_STEPS):
    delta = lap_mean[t] - ef_mean[t]
    print(f"{t+1:5d} {lap_mean[t]:12.4f} {ef_mean[t]:12.4f} {delta:+10.4f}")

print(f"\n--- Recovery Analysis ---")
print(f"Laplacian:")
print(f"  Recovery rate (>0.01): {(recovery_lap > 0.01).sum()}/{N_ROLLOUTS} ({(recovery_lap > 0.01).mean()*100:.1f}%)")
print(f"  Mean recovery magnitude: {recovery_lap.mean():.4f} +/- {recovery_lap.std():.4f}")
print(f"  Median recovery: {np.median(recovery_lap):.4f}")

print(f"\nEdge/Freq (FAIR, 30ep):")
print(f"  Recovery rate (>0.01): {(recovery_ef > 0.01).sum()}/{N_ROLLOUTS} ({(recovery_ef > 0.01).mean()*100:.1f}%)")
print(f"  Mean recovery magnitude: {recovery_ef.mean():.4f} +/- {recovery_ef.std():.4f}")
print(f"  Median recovery: {np.median(recovery_ef):.4f}")

print(f"\n--- Statistical Tests (Final Step SSIM) ---")
print(f"  Laplacian final SSIM:  {final_lap.mean():.4f} +/- {final_lap.std():.4f}")
print(f"  Edge/Freq final SSIM:  {final_ef.mean():.4f} +/- {final_ef.std():.4f}")
print(f"  Paired t-test:   t={t_stat:.4f}, p={t_pval:.2e}")
print(f"  Wilcoxon test:   W={w_stat:.1f}, p={w_pval:.2e}")
print(f"  Cohen's d:       {cohens_d:.4f}")

d_abs = abs(cohens_d)
if d_abs < 0.2:
    d_label = "negligible"
elif d_abs < 0.5:
    d_label = "small"
elif d_abs < 0.8:
    d_label = "medium"
else:
    d_label = "large"
print(f"  Effect size: {d_label}")

## 6. SSIM Curves and Recovery Distributions

Two-panel figure:

**Left: SSIM over rollout steps.** Mean with 95% CI shading for both models. Blue = Laplacian, Red = Edge/Freq. Green shading marks the recovery zone if present.

**Right: Recovery magnitude distributions.** Per-sequence recovery histograms for both models, with a 0.01 threshold line. Distribution near zero = monotonic decay. Right-shifted tail = frequent recovery events.

Saves `fig_ablation_fair.{pdf,png}` to `paper/figures/`.

If both curves look similar, edge/freq losses at these weights (0.01, 0.005) don't meaningfully alter recovery dynamics at 30 epochs. If the Laplacian shows dip-then-rise while Edge/Freq decays monotonically, spatial smoothness enables recovery while edge sharpness suppresses it.

In [ ]:
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.labelsize': 13,
    'axes.titlesize': 14,
    'figure.dpi': 150,
})

C_LAP = '#1565C0'    # blue for Laplacian
C_EF  = '#E53935'    # red for Edge/Freq

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

# ---- Panel 1: SSIM curves ----
ax1.fill_between(steps, lap_mean - lap_ci95, lap_mean + lap_ci95,
                 alpha=0.15, color=C_LAP)
ax1.plot(steps, lap_mean, 'o-', color=C_LAP, linewidth=2.5, markersize=5,
         label='Laplacian only (30 ep)', zorder=5)

ax1.fill_between(steps, ef_mean - ef_ci95, ef_mean + ef_ci95,
                 alpha=0.15, color=C_EF)
ax1.plot(steps, ef_mean, 's-', color=C_EF, linewidth=2.5, markersize=5,
         label='+ Edge/Freq losses (30 ep)', zorder=5)

# Shade recovery zones if detectable
# Find Laplacian recovery zone
lap_min_step = np.argmin(lap_mean[:12])
lap_max_after = lap_min_step + 1 + np.argmax(lap_mean[lap_min_step+1:15])
if lap_mean[lap_max_after] - lap_mean[lap_min_step] > 0.01:
    ax1.axvspan(steps[lap_min_step] - 0.3, steps[lap_max_after] + 0.3,
                alpha=0.06, color='#2E7D32', zorder=0)
    ax1.annotate(
        f'Recovery\n+{lap_mean[lap_max_after]-lap_mean[lap_min_step]:.3f}',
        xy=(steps[lap_max_after], lap_mean[lap_max_after]),
        xytext=(steps[lap_max_after]+1.5, lap_mean[lap_max_after]+0.06),
        fontsize=10, fontweight='bold', color='#2E7D32',
        arrowprops=dict(arrowstyle='->', color='#2E7D32', lw=1.5),
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#E8F5E9',
                  edgecolor='#2E7D32', alpha=0.9))

ax1.set_xlabel('Autoregressive Rollout Step')
ax1.set_ylabel('SSIM (higher = better)')
ax1.set_title('SSIM over Rollout Steps (Fair Comparison)', fontweight='bold')
ax1.legend(fontsize=10, loc='best')
ax1.set_xlim(0.3, ROLLOUT_STEPS + 0.7)
ax1.grid(True, alpha=0.25)

# ---- Panel 2: Recovery distribution ----
bins = np.linspace(-0.05, max(recovery_lap.max(), recovery_ef.max()) + 0.02, 40)
ax2.hist(recovery_lap, bins=bins, alpha=0.6, color=C_LAP, edgecolor='white',
         linewidth=0.5, label=f'Laplacian (mean={recovery_lap.mean():.3f})')
ax2.hist(recovery_ef, bins=bins, alpha=0.6, color=C_EF, edgecolor='white',
         linewidth=0.5, label=f'Edge/Freq (mean={recovery_ef.mean():.3f})')
ax2.axvline(0.01, color='gray', linestyle='--', alpha=0.5, label='Recovery threshold')

ax2.set_xlabel('Recovery Magnitude (SSIM)')
ax2.set_ylabel('Count')
ax2.set_title('Recovery Distribution (Fair Comparison)', fontweight='bold')
ax2.legend(fontsize=9, loc='best')
ax2.grid(True, alpha=0.25)

plt.tight_layout()

# Save
fig_dir = PROJECT / 'paper' / 'figures'
fig_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(str(fig_dir / 'fig_ablation_fair.pdf'), bbox_inches='tight')
fig.savefig(str(fig_dir / 'fig_ablation_fair.png'), dpi=200, bbox_inches='tight')
plt.show()
print(f"Saved to {fig_dir / 'fig_ablation_fair.pdf'} and .png")

## 7. Sample Rollout Comparison

Visual grid comparing individual predicted frames from both models against ground truth. 4 sequences at 5 time steps (1, 5, 10, 15, 19): GT / Laplacian / Edge/Freq rows.

The Laplacian checkpoint is loaded temporarily, rolled out on the first 4 sequences, then freed.

Why bother with visual inspection? Aggregate metrics compress spatial information into single numbers. Two models with identical mean SSIM can produce qualitatively different outputs: globally blurry but structurally coherent vs locally sharp but structurally fragmented.

What to look for:
- **Blur level**: Laplacian predictions may have softer edges; Edge/Freq should be sharper. Check digit boundaries at steps 10-19.
- **Structural coherence at late steps**: do predictions still resemble digits at steps 15-19, or have they become blobs?
- **Recovery signatures**: a degraded image at step 10 but clearer at step 15 is visible recovery.
- **Artifacts**: checkerboard patterns, edge ringing, or mode collapse (all predictions converging to the same average frame).

Saves `fig_ablation_fair_samples.{pdf,png}` to `paper/figures/`.

In [ ]:
# We need Laplacian predictions for the sample comparison.
# Re-load and run on 4 samples only.
model_lap = FluidWorldModelV2(
    in_channels=1, d_model=128, stimulus_dim=1,
    n_encoder_layers=3, max_steps_encoder=6,
    belief_spatial_hw=16, n_belief_evolve=3,
    loss_type="bce", use_fatigue=False, use_inhibition=True,
    use_memory_pump=True, use_hebbian=True, use_deltanet=True, use_titans=True,
).to(device)

CKPT_LAPLACIAN = str(PROJECT / "checkpoints" / "moving_mnist" / "model_epoch_30.pt")
ckpt_lap = torch.load(CKPT_LAPLACIAN, map_location=device, weights_only=False)
model_lap.load_state_dict(ckpt_lap['model_state_dict'])
model_lap.eval()

# Run on first 4 sequences
n_samples = 4
sample_batch = torch.from_numpy(eval_data[:n_samples]).unsqueeze(2).to(device)
x_init = sample_batch[:, 0]
stim = torch.zeros(n_samples, 1, device=device)

with torch.no_grad():
    lap_preds = model_lap.rollout(x_init, stim, n_steps=ROLLOUT_STEPS).cpu().numpy()

del model_lap
torch.cuda.empty_cache()

# Edge/Freq predictions were saved during rollout
ef_preds = sample_preds[:n_samples]

# Ground truth
gt = eval_data[:n_samples]  # (4, 20, 64, 64)

# Plot
show_steps = [0, 4, 9, 14, 18]  # steps 1, 5, 10, 15, 19
n_cols = len(show_steps)
n_rows = n_samples * 3  # GT, Laplacian, Edge/Freq per sample

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 1.8, n_rows * 1.8))

for s in range(n_samples):
    for col, t in enumerate(show_steps):
        # Ground truth (frame t+1 is the target for rollout step t)
        ax_gt = axes[s * 3, col]
        ax_gt.imshow(gt[s, t + 1], cmap='gray', vmin=0, vmax=1)
        ax_gt.axis('off')
        if col == 0:
            ax_gt.set_ylabel(f'Seq {s+1}\nGT', fontsize=9, rotation=0,
                             labelpad=40, va='center')
        if s == 0:
            ax_gt.set_title(f'Step {t+1}', fontsize=10, fontweight='bold')

        # Laplacian prediction
        ax_lap = axes[s * 3 + 1, col]
        # lap_preds shape: (n_samples, ROLLOUT_STEPS, 1, 64, 64)
        img_lap = lap_preds[s, t, 0] if lap_preds.ndim == 5 else lap_preds[s, t]
        ax_lap.imshow(img_lap, cmap='gray', vmin=0, vmax=1)
        ax_lap.axis('off')
        if col == 0:
            ax_lap.set_ylabel('Laplacian', fontsize=9, rotation=0,
                              labelpad=40, va='center', color=C_LAP)

        # Edge/Freq prediction
        ax_ef = axes[s * 3 + 2, col]
        img_ef = ef_preds[s, t, 0] if ef_preds.ndim == 5 else ef_preds[s, t]
        ax_ef.imshow(img_ef, cmap='gray', vmin=0, vmax=1)
        ax_ef.axis('off')
        if col == 0:
            ax_ef.set_ylabel('Edge/Freq', fontsize=9, rotation=0,
                             labelpad=40, va='center', color=C_EF)

    # Add separator line between samples (except after last)
    if s < n_samples - 1:
        for col in range(n_cols):
            axes[s * 3 + 2, col].axhline(y=63, color='gray', linewidth=2,
                                          xmin=-0.1, xmax=1.1, clip_on=False)

plt.suptitle('Sample Rollout Comparison: GT vs Laplacian vs Edge/Freq (both 30 epochs)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()

fig.savefig(str(fig_dir / 'fig_ablation_fair_samples.pdf'), bbox_inches='tight')
fig.savefig(str(fig_dir / 'fig_ablation_fair_samples.png'), dpi=200, bbox_inches='tight')
plt.show()
print(f"Saved to {fig_dir / 'fig_ablation_fair_samples.pdf'} and .png")

## 8. Save All Results

Everything goes to `experiments/analysis/edgefreq_fair_stats.npz`:

| Key | Shape | Description |
|-----|-------|-------------|
| `ssim_matrix` | (500, 19) | Edge/Freq per-step SSIM |
| `mse_matrix` | (500, 19) | Edge/Freq per-step MSE |
| `recovery_ef` | (500,) | Edge/Freq per-sequence recovery magnitude |
| `ssim_laplacian` | (500, 19) | Laplacian per-step SSIM |
| `mse_laplacian` | (500, 19) | Laplacian per-step MSE |
| `recovery_lap` | (500,) | Laplacian per-sequence recovery magnitude |
| `t_stat`, `t_pval` | scalar | Paired t-test (final step) |
| `w_stat`, `w_pval` | scalar | Wilcoxon (final step) |
| `cohens_d` | scalar | Cohen's d (final step) |

This file is the single source of truth for fair ablation results. Downstream analyses should load from here rather than re-running rollouts.

In [ ]:
save_path = PROJECT / 'experiments' / 'analysis' / 'edgefreq_fair_stats.npz'
np.savez(
    str(save_path),
    # Edge/Freq FAIR (30ep) results
    ssim_matrix=ssim_matrix,
    mse_matrix=mse_matrix,
    recovery_ef=recovery_ef,
    # Laplacian (30ep) results (copied for convenience)
    ssim_laplacian=ssim_laplacian,
    mse_laplacian=mse_laplacian,
    recovery_lap=recovery_lap,
    # Statistical tests
    t_stat=t_stat,
    t_pval=t_pval,
    w_stat=w_stat,
    w_pval=w_pval,
    cohens_d=cohens_d,
    # Metadata
    n_rollouts=N_ROLLOUTS,
    rollout_steps=ROLLOUT_STEPS,
)
print(f"Results saved to: {save_path}")
print(f"Contents: {list(np.load(str(save_path)).keys())}")